### Without Agent

In [3]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="gpt-oss:20b")

response = model.invoke([{"role": "user", "content": "What's the weather in Amman?"}])

response.content

'I’m sorry, but I don’t have the ability to pull live weather data.  \n\nFor the most up‑to‑date forecast in Amman, you can check:\n\n- **Weather websites** such as [Weather.com](https://weather.com/), [AccuWeather](https://www.accuweather.com/), or the local “Jordan Meteorological Department” site.  \n- **Weather apps** on your phone (Apple Weather, Google Weather, Weather Underground, etc.).  \n- **Search engines**: typing “Amman weather” into Google or Bing will give a quick snapshot plus a 7‑day outlook.\n\nIf you let me know the date you’re interested in, I can give you a typical climate pattern for that time of year!'

### With Agent

In [ ]:
import requests
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_ollama import ChatOllama


@tool
def get_weather(city: str) -> str:
    # The docstring below is what the model reads to understand this tool
    """Get the current weather for a city.""" # used by the model to understand what this tool does
    # 1. Find the city's coordinates
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1},
    ).json()
    if not geo.get("results"):
        return f"I couldn't find a city called {city}."
    place = geo["results"][0]

    # 2. Get the current weather there
    weather = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": place["latitude"],
            "longitude": place["longitude"],
            "current": "temperature_2m,wind_speed_10m,relative_humidity_2m",
        },
    ).json()["current"]

    return (
        f"{place['name']}, {place.get('country', '')}: "
        f"{weather['temperature_2m']}°C, humidity {weather['relative_humidity_2m']}%, "
        f"wind {weather['wind_speed_10m']} km/h"
    )


model = ChatOllama(model="gpt-oss:20b")
agent = create_agent(model=model, tools=[get_weather])

while True:
    question = input("You (press Enter to stop): ").strip()
    if question.lower() in ("", "exit", "quit"):
        print("Bye!")
        break
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print("Agent:", result["messages"][-1].content)

Agent: **Amman, Jordan**  
- Temperature: **18.8 °C**  
- Humidity: **43 %**  
- Wind: **3.1 km/h**
Bye!
